In [2]:
import sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from sklearn.metrics import precision_recall_fscore_support, average_precision_score

from src.functions.data_loader import load_raw
from src.functions.models import GCN, SAGE, MLP

features, classes, edges = load_raw()
df = features.merge(classes, on="txId", how="left")

id_map = {tx: i for i, tx in enumerate(df["txId"])}
src = edges["txId1"].map(id_map).values
dst = edges["txId2"].map(id_map).values
edge_index = to_undirected(torch.tensor(np.array([src, dst]), dtype=torch.long))

feat_cols = [c for c in df.columns if c.startswith("feature_")]
x = torch.tensor(df[feat_cols].values, dtype=torch.float)
y = torch.tensor(df["class"].map({"1": 1, "2": 0}).fillna(-1).astype(int).values)
time = torch.tensor(df["time_step"].values)

data = Data(x=x, edge_index=edge_index, y=y)
print(data)

Data(x=[203769, 165], edge_index=[2, 468710], y=[203769])


In [3]:
train_m = (y >= 0) & (time <= 24)
val_m   = (y >= 0) & (time >= 25) & (time <= 34)
test_mask = (y >= 0) & (time > 34)

print("Train:", train_m.sum().item(), "Val:", val_m.sum().item(), "Test:", test_mask.sum().item())
print("Illicit - train:", (y[train_m] == 1).sum().item(),
      "val:", (y[val_m] == 1).sum().item(), "test:", (y[test_mask] == 1).sum().item())

same_step = (time[edge_index[0]] == time[edge_index[1]]).float().mean().item()
print("Share of edges within the same time step:", same_step)

Train: 23606 Val: 6288 Test: 16670
Illicit - train: 2219 val: 1243 test: 1083
Share of edges within the same time step: 1.0


In [4]:
def run(model_cls, seed, max_epochs=400, patience=40):
    torch.manual_seed(seed)
    m = model_cls(x.shape[1])
    opt = torch.optim.Adam(m.parameters(), lr=0.01, weight_decay=5e-4)
    w = torch.tensor([1.0, (y[train_m] == 0).sum().item() / (y[train_m] == 1).sum().item()])
    best_f1, best_state, wait = -1, None, 0
    for epoch in range(max_epochs):
        m.train(); opt.zero_grad()
        out = m(data.x, data.edge_index)
        F.cross_entropy(out[train_m], data.y[train_m], weight=w).backward()
        opt.step()
        m.eval()
        with torch.no_grad():
            pv = m(data.x, data.edge_index).argmax(1)
        _, _, f, _ = precision_recall_fscore_support(
            data.y[val_m].numpy(), pv[val_m].numpy(), pos_label=1, average="binary", zero_division=0)
        if f > best_f1:
            best_f1, wait = f, 0
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    m.load_state_dict(best_state); m.eval()
    with torch.no_grad():
        logits = m(data.x, data.edge_index)
    pred = logits.argmax(1)[test_mask].numpy()
    prob = torch.softmax(logits, 1)[:, 1][test_mask].numpy()
    yt = data.y[test_mask].numpy()
    p, r, f1, _ = precision_recall_fscore_support(yt, pred, pos_label=1, average="binary", zero_division=0)
    return p, r, f1, average_precision_score(yt, prob)

In [5]:
import time as t
start = t.time()
print("GCN seed 0:", np.round(run(GCN, 0), 3), round(t.time() - start, 1), "s")

GCN seed 0: [0.177 0.771 0.288 0.319] 31.2 s


In [6]:
n_seeds = 3  
results = {}
for name, cls in [("MLP (no graph)", MLP), ("GCN", GCN), ("GraphSAGE", SAGE)]:
    res = np.array([run(cls, s) for s in range(n_seeds)])
    results[name] = res
    print(name, "mean (P, R, F1, PR-AUC):", res.mean(0).round(3), "std:", res.std(0).round(3))

MLP (no graph) mean (P, R, F1, PR-AUC): [0.406 0.631 0.494 0.337] std: [0.006 0.013 0.004 0.009]
GCN mean (P, R, F1, PR-AUC): [0.188 0.766 0.302 0.29 ] std: [0.012 0.004 0.014 0.023]
GraphSAGE mean (P, R, F1, PR-AUC): [0.34  0.693 0.454 0.453] std: [0.04  0.015 0.032 0.036]


In [7]:
import os, json

Xn = df[feat_cols].values
yn = y.numpy()
tr, te = train_m.numpy(), test_mask.numpy()

os.makedirs("../results", exist_ok=True)
np.savez("../results/split_1_24.npz", X=Xn, y=yn, tr=tr, te=te)